# Notebook 6 ? Bagi Dataset

Pisahkan satu CSV menjadi dua atau tiga bagian acak, misalnya data latih dan uji. Rasio dapat diatur dan hasil dapat diulang dengan angka acak yang sama. Gunakan `GROUP_COLUMN` bila semua baris dari satu orang/dokumen harus tetap berada di bagian yang sama.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# KONFIGURASI
INPUT_PATH = Path("Input/input.csv")
SEP = ","
TRAIN_RATIO = 0.8
VALIDATION_RATIO = 0.0  # Contoh: 0.1; TEST akan mengambil sisa proporsi
GROUP_COLUMN = None     # Contoh: "document_id" untuk membagi per grup, bukan per baris
RANDOM_SEED = 42
OUTPUT_DIR = Path("Output")
def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Buat dan simpan bagian dataset

In [ ]:
if TRAIN_RATIO <= 0 or VALIDATION_RATIO < 0 or TRAIN_RATIO + VALIDATION_RATIO >= 1:
    raise ValueError("TRAIN_RATIO harus > 0 dan TRAIN_RATIO + VALIDATION_RATIO harus < 1.")
source = INPUT_PATH if INPUT_PATH.is_absolute() else BASE_DIR / INPUT_PATH
if not source.exists(): raise FileNotFoundError(f"File tidak ditemukan: {source}")
df = pd.read_csv(source, sep=SEP, dtype="string", encoding="utf-8-sig")
if GROUP_COLUMN:
    if GROUP_COLUMN not in df: raise KeyError(f"Kolom grup tidak ditemukan: {GROUP_COLUMN}")
    if df[GROUP_COLUMN].isna().any(): raise ValueError("Isi GROUP_COLUMN yang kosong terlebih dahulu agar semua baris bisa dibagi.")
    groups = df[GROUP_COLUMN].unique().tolist()
    rng = np.random.default_rng(RANDOM_SEED)
    rng.shuffle(groups)
    n_train = int(len(groups) * TRAIN_RATIO)
    n_val = int(len(groups) * VALIDATION_RATIO)
    train_groups, val_groups = set(groups[:n_train]), set(groups[n_train:n_train+n_val])
    train = df[df[GROUP_COLUMN].isin(train_groups)]
    val = df[df[GROUP_COLUMN].isin(val_groups)]
    test = df[~df[GROUP_COLUMN].isin(train_groups | val_groups)]
else:
    rng = np.random.default_rng(RANDOM_SEED)
    order = rng.permutation(len(df))
    n_train = int(len(df) * TRAIN_RATIO)
    n_val = int(len(df) * VALIDATION_RATIO)
    train = df.iloc[order[:n_train]]
    val = df.iloc[order[n_train:n_train+n_val]]
    test = df.iloc[order[n_train+n_val:]]

outdir = OUTPUT_DIR if OUTPUT_DIR.is_absolute() else BASE_DIR / OUTPUT_DIR
outdir.mkdir(parents=True, exist_ok=True)
for name, part in [("train", train), ("validation", val), ("test", test)]:
    if name == "validation" and VALIDATION_RATIO == 0: continue
    target = outdir / f"{name}.csv"
    part.to_csv(target, index=False, encoding="utf-8-sig")
    print(f"{name}: {len(part):,} baris -> {target}")
print("Pembagian grup menjaga satu grup tetap di satu bagian; rasio akhir dapat sedikit berbeda karena ukuran grup.")
